In [1]:
import pandas as pd
import numpy as np
import re

from datetime import datetime
from pathlib import Path

from sqlalchemy.orm import Session
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

In [2]:
import sys

ROOT_DIR = r"D:\ESM"

if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(ROOT_DIR)

D:\ESM


In [29]:
from app.core.config import settings
from app.core.database import Base

from app.models.user import User
from app.models.department import Department
from app.models.role import UserRole
from app.models.user_type import UserType
from app.models.mst_status import Status

from app.models.risk_register import RiskRegister
from app.models.risk_description import RiskDescription
from app.models.risk_treatment import RiskTreatment

from app.services.risk_service import get_color_code

In [4]:
# Excel Path

EXCEL_FILE = r"D:\ESM\Data\1. HR- Risk Register Review_2025_Final.xlsx"

# Department

DEFAULT_DEPT_ID = 20

# Demo Password

DEFAULT_PASSWORD = "123456"

# Created By

CREATED_BY = 1

# Financial Year

FINANCIAL_YEAR = "2026-2027"

# Risk Status

RISK_STATUS = 9

# Approval Status

APPROVAL_STATUS = 12

In [5]:
engine = create_engine(settings.DATABASE_URL)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

db = SessionLocal()

print("Database Connected")

Database Connected


In [6]:
raw_df = pd.read_excel(EXCEL_FILE)

raw_df.head(10)

,S. No,Category,Risk Description,Inherent Risk Level,Current Mitigation,Current\nRisk Level,Risk Owner,Risk Treatment,Unnamed: 8,Unnamed: 9,Q1,Q2,Q3,Q4,Back up document
0,NaN,NaN,NaN,(Impact/ Likelihood),NaN,(Impact/ Likelihood),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Action Plan,Due Date,Action owner,NaN,NaN,NaN,NaN,NaN
2,1.0,Recruitment & Hiring,Delay in acquisition of key Positions leading...,3D,1. Tie-ups with reputed recruitment agencies t...,2C,Shruti N.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Inadequate background checks,3C,1. Continuous review of BGV's for each cases\n...,2C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Attrition risk : Inability to retain people at...,3D,"1. Retention planning for key talent, critical...",3C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2.0,Employee Exp.,Employee Disengagement and Inadequate talent D...,2C,1. SMART goals based Performance Management \...,2B,Abhijit S.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,4.0,Complaiance Mgt.,Non-compliance to applicble labour laws leadin...,3D,1. All applicable labour compliances are mappe...,3C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,5.0,Compensation & Benefits,Risk of overcompensation or undercompensation ...,3D,1. Periodic benchmark of compensation against ...,2C,Shirin V.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
raw_df.columns = [
    "S.No",
    "Category",
    "Risk Description",
    "Inherent Risk",
    "Current Mitigation",
    "Current Risk",
    "Risk Owner",
    # "Risk Treatment",
    "Action Plan",
    "Due Date",
    "Action Owner",
    "Q1",
    "Q2",
    "Q3",
    "Q4",
    "Backup"
]

In [8]:
df = raw_df.iloc[2:].copy()

df.reset_index(drop=True, inplace=True)

df.head()

,S.No,Category,Risk Description,Inherent Risk,Current Mitigation,Current Risk,Risk Owner,Action Plan,Due Date,Action Owner,Q1,Q2,Q3,Q4,Backup
0,1.0,Recruitment & Hiring,Delay in acquisition of key Positions leading...,3D,1. Tie-ups with reputed recruitment agencies t...,2C,Shruti N.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Inadequate background checks,3C,1. Continuous review of BGV's for each cases\n...,2C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Attrition risk : Inability to retain people at...,3D,"1. Retention planning for key talent, critical...",3C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.0,Employee Exp.,Employee Disengagement and Inadequate talent D...,2C,1. SMART goals based Performance Management \...,2B,Abhijit S.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4.0,Complaiance Mgt.,Non-compliance to applicble labour laws leadin...,3D,1. All applicable labour compliances are mappe...,3C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df["Category"] = df["Category"].ffill()

df["S.No"] = df["S.No"].ffill()

In [10]:
text_columns = [
    "Category",
    "Risk Description",
    "Current Mitigation",
    "Risk Owner",
    "Action Plan",
    "Action Owner"
]

for col in text_columns:

    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

In [11]:
df.head(10)

,S.No,Category,Risk Description,Inherent Risk,Current Mitigation,Current Risk,Risk Owner,Action Plan,Due Date,Action Owner,Q1,Q2,Q3,Q4,Backup
0,1.0,Recruitment & Hiring,Delay in acquisition of key Positions leading...,3D,1. Tie-ups with reputed recruitment agencies t...,2C,Shruti N.,,NaN,,NaN,NaN,NaN,NaN,NaN
1,1.0,Recruitment & Hiring,Inadequate background checks,3C,1. Continuous review of BGV's for each cases\n...,2C,Manali,,NaN,,NaN,NaN,NaN,NaN,NaN
2,1.0,Recruitment & Hiring,Attrition risk : Inability to retain people at...,3D,"1. Retention planning for key talent, critical...",3C,Manali,,NaN,,NaN,NaN,NaN,NaN,NaN
3,2.0,Employee Exp.,Employee Disengagement and Inadequate talent D...,2C,1. SMART goals based Performance Management \...,2B,Abhijit S.,,NaN,,NaN,NaN,NaN,NaN,NaN
4,4.0,Complaiance Mgt.,Non-compliance to applicble labour laws leadin...,3D,1. All applicable labour compliances are mappe...,3C,Manali,,NaN,,NaN,NaN,NaN,NaN,NaN
5,5.0,Compensation & Benefits,Risk of overcompensation or undercompensation ...,3D,1. Periodic benchmark of compensation against ...,2C,Shirin V.,,NaN,,NaN,NaN,NaN,NaN,NaN


In [12]:
df.groupby(["S.No", "Category"]).size()

S.No  Category               
1.0   Recruitment & Hiring       3
2.0   Employee Exp.              1
4.0   Complaiance Mgt.           1
5.0   Compensation & Benefits    1
dtype: int64

In [13]:
grouped = df.groupby(["S.No", "Category"])

for (sno, category), group in grouped:

    print("=" * 50)
    print("Risk :", category)

    print(group[[
        "Risk Description",
        "Risk Owner"
    ]])

Risk : Recruitment & Hiring
                                    Risk Description Risk Owner
0  Delay  in acquisition of key Positions leading...  Shruti N.
1                       Inadequate background checks     Manali
2  Attrition risk : Inability to retain people at...     Manali
Risk : Employee Exp.
                                    Risk Description  Risk Owner
3  Employee Disengagement and Inadequate talent D...  Abhijit S.
Risk : Complaiance Mgt.
                                    Risk Description Risk Owner
4  Non-compliance to applicble labour laws leadin...     Manali
Risk : Compensation & Benefits
                                    Risk Description Risk Owner
5  Risk of overcompensation or undercompensation ...  Shirin V.


In [14]:
risk = {
    "category": category,
    "rows": group
}

In [15]:
def clean_text(value):
    """
    Convert NaN to empty string and trim spaces.
    """

    if pd.isna(value):
        return ""

    return str(value).strip()


def split_name(full_name):
    """
    Split full name into first name and last name.
    """

    full_name = clean_text(full_name)

    if not full_name:
        return "", ""

    parts = full_name.split()

    first_name = parts[0]

    last_name = " ".join(parts[1:]) if len(parts) > 1 else ""

    return first_name, last_name


def create_log_id(email):
    """
    Use email as login id.
    """

    email = clean_text(email)

    return email.lower()


def clean_email(email):
    """
    Remove spaces and convert to lowercase.
    """

    return clean_text(email).lower()

In [16]:
impact_reverse = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5
}


def parse_risk_code(code):

    code = clean_text(code).upper()

    if code == "":
        return None, None

    match = re.match(r"([1-5])([A-E])", code)

    if not match:
        raise ValueError(f"Invalid Risk Code : {code}")

    likelihood = int(match.group(1))

    impact = impact_reverse[match.group(2)]

    return likelihood, impact

In [17]:
impact_map = {
    1: "A",
    2: "B",
    3: "C",
    4: "D",
    5: "E"
}


def build_color_code(likelihood, impact):

    if likelihood is None or impact is None:
        return None

    return f"{likelihood}{impact_map.get(impact)}"


def get_color(code):
    """
    Wrapper for your existing function.
    """

    return get_color_code(code)

In [18]:
def parse_date(value):

    if pd.isna(value):
        return None

    if isinstance(value, datetime):
        return value

    return pd.to_datetime(value)


def current_time():

    return datetime.now()

In [19]:
def generate_risk_id(db: Session, dept_id: int):

    dept = (
        db.query(Department)
        .filter(Department.id == dept_id)
        .with_for_update()
        .first()
    )

    if not dept:
        raise Exception("Department not found")

    dept.last_risk_number += 1

    number = dept.last_risk_number

    risk_id = f"{dept.dept_short_name}-{str(number).zfill(4)}"

    return risk_id

In [20]:
print(parse_date("2026-07-10"))

print(current_time())

print(generate_risk_id(db, DEFAULT_DEPT_ID))

2026-07-10 00:00:00
2026-06-29 17:19:33.982725


NameError: name 'Department' is not defined

Make User Data

In [22]:
def prepare_master_data(df, db):

    print("="*60)
    print("Preparing Master Data")
    print("="*60)

    users = set()

    # Risk Owner
    users.update(
        df["Risk Owner"]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    # Action Owner
    users.update(
        df["Action Owner"]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    users.discard("")

    print(f"Total Users : {len(users)}")

    return sorted(users)

In [37]:
users = prepare_master_data(df, db)

users

Preparing Master Data
Total Users : 4


['Abhijit S.', 'Manali', 'Shirin V.', 'Shruti N.']

In [39]:
def get_user_by_email(db, email):

    return (
        db.query(User)
        .filter(
            User.email.ilike(email.strip())
        )
        .first()
    )

In [40]:
def get_user_by_name(db, full_name):

    first_name, last_name = split_name(full_name)

    return (
        db.query(User)
        .filter(
            User.first_name.ilike(first_name),
            User.last_name.ilike(last_name)
        )
        .first()
    )

In [ ]:
def check_user_exists(db, email=None, full_name=None):
    """
    Check whether a user already exists.
    Priority:
    1. Email
    2. Full Name
    """

    # Search by email
    if email and str(email).strip():

        user = (
            db.query(User)
            .filter(User.email.ilike(email.strip()))
            .first()
        )

        if user:
            print(f"✓ User found by Email : {user.email}")
            return user

    # Search by name
    if full_name and str(full_name).strip():

        first_name, last_name = split_name(full_name)

        user = (
            db.query(User)
            .filter(
                User.first_name.ilike(first_name),
                User.last_name.ilike(last_name)
            )
            .first()
        )

        if user:
            print(f"✓ User found by Name : {full_name}")
            return user

    print("✗ User not found")

    return None

In [50]:
user = check_user_exists(
    db,
    email="shruti.nair@gmail.com",
    full_name="Shruti Nair"
)

print(user)

✓ User found by Email : shruti.nair@gmail.com


In [47]:
def create_user(
    db: Session,
    first_name,
    last_name,
    email,
    dept_id,
    role_id,
    user_type_id,
    status="Active",
):

    user = User(
        log_id=email.lower(),
        password="123456",
        first_name=first_name,
        last_name=last_name,
        email=email.lower(),

        dept_id=dept_id,
        role_id=role_id,
        user_type_id=user_type_id,

        status=status,

        address=None,
        contact_no=None,
        country=None,
        country_code=None,
        std_code=None,
        user_city=None,
        photo=None,

        created_by=1,
        created_on=datetime.utcnow(),
        is_deleted=0
    )

    db.add(user)

    db.flush()          # Get ID without committing

    db.refresh(user)

    return user

In [ ]:
user = check_user_exists(db=db,email=email,full_name=name
)

if user is None:

    user = create_new_user(
        db=db,
        full_name=name,
        email=email,
        dept_id=dept_id,
        role_id=role_id,
        user_type_id=user_type_id
    )

print(user.id)

user = get_or_create_user(
    db=db,
    full_name="Shruti Nair",
    email="shruti.nair@gmail.com",
    dept_id=20,
    role_id=26,
    user_type_id=7
)

print(user.id)
print(user.first_name)
print(user.email)

NameError: name 'email' is not defined